In [2]:
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

df = pd.read_csv("../datasets/reddit.csv")
df.rename(columns={'clean_text': 'message', 'is_depression': 'label'}, inplace=True)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(df["message"], df['label'], test_size=0.3)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

classification_report_result = classification_report(y_test, y_pred)
print(classification_report_result)

              precision    recall  f1-score   support

           0       0.98      0.54      0.70      1154
           1       0.68      0.99      0.81      1166

    accuracy                           0.77      2320
   macro avg       0.83      0.76      0.75      2320
weighted avg       0.83      0.77      0.75      2320



In [4]:
from xgboost import XGBClassifier


xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', xgb_model)
])
xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)

classification_report_result = classification_report(y_test, y_pred_xgb)
print(classification_report_result)

/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [01:45:42] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1154
           1       0.98      0.95      0.97      1166

    accuracy                           0.97      2320
   macro avg       0.97      0.97      0.97      2320
weighted avg       0.97      0.97      0.97      2320



In [5]:
from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(n_jobs=-1, reg_lambda=0.1, num_leaves=64, max_depth=-1, learning_rate=0.1, n_estimators=100)
lgbm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', lgbm_model)
])
lgbm_pipeline.fit(X_train, y_train)
y_pred_lgbm = lgbm_pipeline.predict(X_test)
classification_report_result = classification_report(y_test, y_pred_lgbm)
print(classification_report_result)

[LightGBM] [Info] Number of positive: 2665, number of negative: 2746
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.081450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 55968
[LightGBM] [Info] Number of data points in the train set: 5411, number of used features: 1346
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.492515 -> initscore=-0.029941
[LightGBM] [Info] Start training from score -0.029941
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1154
           1       0.99      0.95      0.97      1166

    accuracy                           0.97      2320
   macro avg       0.97      0.97      0.97      2320
weighted avg       0.97      0.97      0.97      2320



/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
